# FakeVLM 全流程推理 Notebook（下载代码 → 配置环境 → 下载模型 → 读取图片 → 输出AI生成概率）

> 目标：使用预训练 `lingcco/fakeVLM` 模型，对输入图片计算 **AI 生成（fake）概率** 与 **真实（real）概率**，并导出结果文件。

该 Notebook 设计为“从零可复现”的流程，你可以在本地、服务器或云端（如 JupyterLab）逐步执行。

## 0) 使用说明

- 推荐 GPU 环境（至少 24GB 显存更稳妥，7B 多模态模型较大）。
- 本 Notebook 提供两种安装方式：
  1. **Conda + pip**（更接近项目 README）
  2. **纯 pip**（在已有 Jupyter 环境中更方便）
- 若你已在仓库根目录运行，可跳过 clone 步骤。

In [ ]:
# 可选：查看当前 Python 与 CUDA 环境
import sys, os, platform, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
try:
    out = subprocess.check_output(['nvidia-smi'], text=True)
    print(out.splitlines()[0])
except Exception as e:
    print('nvidia-smi 不可用：', e)

## 1) 下载项目代码

如果你还没有代码仓库，请执行：

In [ ]:
# 仅首次需要
# !git clone https://github.com/opendatalab/FakeVLM.git
# %cd FakeVLM

# 如果你已经在 FakeVLM 根目录，可直接运行
import os
print('当前目录:', os.getcwd())

## 2) 环境配置

### 2.1 Conda 方式（推荐，和项目 README 一致）

In [ ]:
# 在终端执行（Notebook 中展示命令，不强制运行）
conda_cmds = r'''
conda create -n fakevlm python=3.10 -y
conda activate fakevlm
python -m pip install -r requirements.txt
python -m pip install --no-cache-dir --no-build-isolation flash-attn
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name fakevlm --display-name "Python (fakevlm)"
'''
print(conda_cmds)

### 2.2 纯 pip 方式（当前 Notebook 内安装）

> 如果你在托管 Notebook 环境中，没有 conda 权限，可用此方式。

In [ ]:
# 谨慎：大依赖安装耗时较长
# %pip install -r requirements.txt
# %pip install --no-cache-dir --no-build-isolation flash-attn
# %pip install huggingface_hub

## 3) 下载预训练模型

项目说明的已训练权重：`lingcco/fakeVLM`。

你可以：
- 直接让 Transformers 在首次推理时自动下载；或
- 先手动下载到本地目录，再从本地加载。

In [ ]:
# 方案 A：手动下载（推荐，便于复用）
# 如需私有/限流环境，请先 `huggingface-cli login`
from huggingface_hub import snapshot_download

MODEL_REPO = 'lingcco/fakeVLM'
LOCAL_MODEL_DIR = './checkpoints/fakeVLM'

# 首次会下载较久
# local_dir = snapshot_download(repo_id=MODEL_REPO, local_dir=LOCAL_MODEL_DIR, local_dir_use_symlinks=False)
# print('模型已下载到:', local_dir)
print('如需下载，取消注释 snapshot_download 行。')

## 4) 准备待测图片

把你要检测的图片放到一个目录，例如：`./demo_images/`。

下面代码会递归读取常见图片格式。

In [ ]:
from pathlib import Path

IMAGE_DIR = Path('./demo_images')  # 修改为你的图片目录
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

image_paths = [p for p in IMAGE_DIR.rglob('*') if p.suffix.lower() in EXTS]
print(f'找到图片数量: {len(image_paths)}')
for p in image_paths[:10]:
    print('-', p)

## 5) 加载模型并构造“概率预测”

项目原评估脚本主要是“文本生成 + 规则提取 fake/real”。

为得到**概率**，这里采用更稳定的做法：
1. 固定提示词，让模型回答 `real` 或 `fake`；
2. 取首个生成 token 的 logits；
3. 对 `real` 和 `fake` 的 token logits 做 softmax，得到两类概率。

> 注意：多模态 LLM 的 token 粒度可能把单词拆成多个 token。此处使用 tokenizer 编码后取“首 token”近似，实操可用。

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

MODEL_PATH = './checkpoints/fakeVLM'  # 若未手动下载，可改成 'lingcco/fakeVLM'
BASE_PROCESSOR = 'llava-hf/llava-1.5-7b-hf'

processor = AutoProcessor.from_pretrained(BASE_PROCESSOR, revision='a272c74')
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    low_cpu_mem_usage=True,
).to(device).eval()

# 取 real/fake 对应 token id（首 token近似）
real_ids = processor.tokenizer.encode(' real', add_special_tokens=False)
fake_ids = processor.tokenizer.encode(' fake', add_special_tokens=False)
print('real token ids:', real_ids)
print('fake token ids:', fake_ids)

assert len(real_ids) > 0 and len(fake_ids) > 0, 'tokenize real/fake 失败'
REAL_ID = real_ids[0]
FAKE_ID = fake_ids[0]

## 6) 单张图片预测（返回 fake 概率）

In [ ]:
import torch.nn.functional as F

PROMPT = (
    'USER: <image>\n'
    'Please determine whether this image is real or fake. '
    'Answer with one word: real or fake.\n'
    'ASSISTANT:'
)

@torch.no_grad()
def predict_fake_prob(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(
        text=PROMPT,
        images=image,
        return_tensors='pt',
        padding='max_length',
        max_length=1024,
        truncation=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 前向：拿下一token logits
    outputs = model(**inputs)
    next_token_logits = outputs.logits[:, -1, :]  # [1, vocab]

    pair_logits = torch.stack([
        next_token_logits[0, REAL_ID],
        next_token_logits[0, FAKE_ID],
    ])
    probs = F.softmax(pair_logits, dim=0)
    real_prob, fake_prob = probs.tolist()

    pred_label = 'fake' if fake_prob >= real_prob else 'real'
    return {
        'image_path': str(image_path),
        'pred_label': pred_label,
        'fake_probability': float(fake_prob),
        'real_probability': float(real_prob),
    }

# 示例
if image_paths:
    one = predict_fake_prob(image_paths[0])
    print(one)
else:
    print('请先在 IMAGE_DIR 放入图片。')

## 7) 批量预测并导出结果

输出：
- `results/fake_probability_results.csv`
- `results/fake_probability_results.json`

In [ ]:
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json

results = []
for p in tqdm(image_paths, desc='Predicting'):
    try:
        results.append(predict_fake_prob(p))
    except Exception as e:
        results.append({
            'image_path': str(p),
            'pred_label': 'error',
            'fake_probability': None,
            'real_probability': None,
            'error': str(e),
        })

out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(results)
csv_path = out_dir / 'fake_probability_results.csv'
json_path = out_dir / 'fake_probability_results.json'

df.to_csv(csv_path, index=False, encoding='utf-8-sig')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print('CSV:', csv_path.resolve())
print('JSON:', json_path.resolve())
print(df.head(10))

## 8) （可选）生成解释文本

如果你还希望得到模型对伪造痕迹的解释，可用 `generate` 输出完整回答。

In [ ]:
@torch.no_grad()
def generate_explanation(image_path, max_new_tokens=128):
    image = Image.open(image_path).convert('RGB')
    prompt = (
        'USER: <image>\n'
        'Please determine whether this image is real or fake, and explain the artifact clues briefly.\n'
        'ASSISTANT:'
    )
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors='pt',
        padding='max_length',
        max_length=1024,
        truncation=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    text = processor.decode(output[0], skip_special_tokens=True)
    return text

if image_paths:
    print(generate_explanation(image_paths[0], max_new_tokens=80))

## 9) 常见问题排查

1. **显存不足（CUDA out of memory）**
   - 降低 `max_length`、减少 batch（本 Notebook 默认单张）；
   - 使用更大显存 GPU；
   - 尝试 `torch_dtype=torch.bfloat16`（视硬件支持）。

2. **flash-attn 安装失败**
   - 常见于 CUDA/编译环境不匹配，可先不装并去掉相关参数；
   - 或参考 flash-attn 官方编译说明。

3. **模型下载慢/失败**
   - 配置 Hugging Face 镜像或代理；
   - `huggingface-cli login` 后重试。

4. **概率不稳定**
   - 换更明确的 prompt（只允许输出 real/fake）；
   - 用多个 prompt 模板做平均（prompt ensembling）。